In [18]:
import dgl
import torch
import pickle
import numpy as np
from pathlib import Path

def load_saved_graphs(input_path):
    """Load graphs that were saved with save_graphs()."""
    input_path = Path(input_path)

    if not input_path.exists():
        raise FileNotFoundError(f"Graph file not found: {input_path}")

    print(f"Loading graphs from {input_path}")

    # Load graphs
    graph_list, _ = dgl.load_graphs(str(input_path))

    # Load metadata
    metadata_path = input_path.with_suffix('.pkl')
    if not metadata_path.exists():
        raise FileNotFoundError(f"Metadata file not found: {metadata_path}")

    with open(metadata_path, 'rb') as f:
        metadata = pickle.load(f)

    graph_ids = metadata['graph_ids']
    graphs = {graph_id: graph for graph_id, graph in zip(graph_ids, graph_list)}

    print(f"✓ Loaded {len(graphs)} graphs")

    return graphs

# Load the graphs
graphs = load_saved_graphs('my_graphs.bin')

Loading graphs from my_graphs.bin
✓ Loaded 500 graphs


In [19]:
# Explore basic statistics
print(f"Total number of graphs: {len(graphs)}")
print(f"\nFirst 10 graph IDs:")
for i, gid in enumerate(list(graphs.keys())[:500]):
    print(f"  {i+1}. {gid}")

Total number of graphs: 500

First 10 graph IDs:
  1. 21_0
  2. 21_1
  3. 21_2
  4. 21_3
  5. 21_4
  6. 21_5
  7. 21_6
  8. 21_7
  9. 21_8
  10. 21_9
  11. 20_0
  12. 20_1
  13. 20_2
  14. 20_3
  15. 20_4
  16. 20_5
  17. 20_6
  18. 20_7
  19. 20_8
  20. 20_9
  21. 7_0
  22. 7_1
  23. 7_2
  24. 7_3
  25. 7_4
  26. 7_5
  27. 7_6
  28. 7_7
  29. 7_8
  30. 7_9
  31. 6_0
  32. 6_1
  33. 6_2
  34. 6_3
  35. 6_4
  36. 6_5
  37. 6_6
  38. 6_7
  39. 6_8
  40. 6_9
  41. 15_0
  42. 15_1
  43. 15_2
  44. 15_3
  45. 15_4
  46. 15_5
  47. 15_6
  48. 15_7
  49. 15_8
  50. 15_9
  51. 14_0
  52. 14_1
  53. 14_2
  54. 14_3
  55. 14_4
  56. 14_5
  57. 14_6
  58. 14_7
  59. 14_8
  60. 14_9
  61. 36_0
  62. 36_1
  63. 36_2
  64. 36_3
  65. 36_4
  66. 36_5
  67. 36_6
  68. 36_7
  69. 36_8
  70. 36_9
  71. 37_0
  72. 37_1
  73. 37_2
  74. 37_3
  75. 37_4
  76. 37_5
  77. 37_6
  78. 37_7
  79. 37_8
  80. 37_9
  81. 41_0
  82. 41_1
  83. 41_2
  84. 41_3
  85. 41_4
  86. 41_5
  87. 41_6
  88. 41_7
  89. 41_8
 

In [20]:
# Pick a sample graph to explore
sample_graph_id = list(graphs.keys())[0]
g = graphs[sample_graph_id]

print(f"Sample Graph: {sample_graph_id}")
print(f"="*50)
print(f"Number of nodes: {g.num_nodes()}")
print(f"Number of edges: {g.num_edges()}")
print(f"\nNode features: {list(g.ndata.keys())}")
print(f"Edge features: {list(g.edata.keys())}")

Sample Graph: 21_0
Number of nodes: 10
Number of edges: 100

Node features: ['Num_Samples', 'Epidemic_Peak', 'Accumulated_Infections', 'Source_Sink_Score', 'Peak_Timing', 'R0', 'Initial_Population']
Edge features: ['migration_rate']


In [21]:
# Examine node features in detail
print("Node Feature Shapes and Sample Values:")
print("="*50)
for feature_name in g.ndata.keys():
    feature_data = g.ndata[feature_name]
    print(f"\n{feature_name}:")
    print(f"  Shape: {feature_data.shape}")
    print(f"  Data type: {feature_data.dtype}")
    print(f"  First 5 values: {feature_data[:5].numpy()}")
    
    # Count NaN values
    nan_mask = torch.isnan(feature_data)
    nan_count = nan_mask.sum().item()
    print(f"  NaN count: {nan_count} / {len(feature_data)}")
    
    # Compute statistics ignoring NaN
    if nan_count < len(feature_data):
        valid_data = feature_data[~nan_mask]
        print(f"  Min: {valid_data.min():.4f}, Max: {valid_data.max():.4f}, Mean: {valid_data.mean():.4f}")
    else:
        print(f"  All values are NaN")

Node Feature Shapes and Sample Values:

Num_Samples:
  Shape: torch.Size([10])
  Data type: torch.float32
  First 5 values: [43. 25. 26. 42. 38.]
  NaN count: 0 / 10
  Min: 9.0000, Max: 43.0000, Mean: 25.0000

Epidemic_Peak:
  Shape: torch.Size([10])
  Data type: torch.float32
  First 5 values: [3752. 1975. 2675. 3392. 2474.]
  NaN count: 0 / 10
  Min: 438.0000, Max: 3752.0000, Mean: 2028.7000

Accumulated_Infections:
  Shape: torch.Size([10])
  Data type: torch.float32
  First 5 values: [8632. 5203. 6265. 9281. 6853.]
  NaN count: 0 / 10
  Min: 1903.0000, Max: 9281.0000, Mean: 5376.2998

Source_Sink_Score:
  Shape: torch.Size([10])
  Data type: torch.float32
  First 5 values: [ 0.1866171  -0.16734694  0.39482415  0.27491638  0.2999263 ]
  NaN count: 0 / 10
  Min: -0.5455, Max: 0.3948, Mean: -0.0190

Peak_Timing:
  Shape: torch.Size([10])
  Data type: torch.float32
  First 5 values: [121.97407 131.73724 108.03826 139.73438 151.8182 ]
  NaN count: 0 / 10
  Min: 84.1248, Max: 151.8182, M

In [22]:
# Examine edge features in detail
print("\nEdge Feature Shapes and Sample Values:")
print("="*50)
for feature_name in g.edata.keys():
    feature_data = g.edata[feature_name]
    print(f"\n{feature_name}:")
    print(f"  Shape: {feature_data.shape}")
    print(f"  Data type: {feature_data.dtype}")
    print(f"  First 10 values: {feature_data[:10].numpy()}")
    
    # Count NaN values
    nan_mask = torch.isnan(feature_data)
    nan_count = nan_mask.sum().item()
    print(f"  NaN count: {nan_count} / {len(feature_data)}")
    
    # Compute statistics ignoring NaN
    if nan_count < len(feature_data):
        valid_data = feature_data[~nan_mask]
        print(f"  Min: {valid_data.min():.6f}, Max: {valid_data.max():.6f}, Mean: {valid_data.mean():.6f}")
    else:
        print(f"  All values are NaN")


Edge Feature Shapes and Sample Values:

migration_rate:
  Shape: torch.Size([100])
  Data type: torch.float32
  First 10 values: [       nan 0.00020794 0.00011086 0.00012509 0.0003088  0.00024537
 0.00073306 0.00087843 0.00021623 0.00070944]
  NaN count: 10 / 100
  Min: 0.000111, Max: 0.001000, Mean: 0.000557


In [23]:
# Examine graph structure
print("\nGraph Structure:")
print("="*50)

# Get edges
src, dst = g.edges()
print(f"First 10 edges (src -> dst):")
for i in range(min(10, g.num_edges())):
    print(f"  {src[i].item()} -> {dst[i].item()}")

# Check for self-loops
self_loops = (src == dst).sum().item()
print(f"\nNumber of self-loops: {self_loops}")

# Degree distribution
in_degrees = g.in_degrees()
out_degrees = g.out_degrees()
print(f"\nDegree statistics:")
print(f"  In-degree  - Min: {in_degrees.min()}, Max: {in_degrees.max()}, Mean: {in_degrees.float().mean():.2f}")
print(f"  Out-degree - Min: {out_degrees.min()}, Max: {out_degrees.max()}, Mean: {out_degrees.float().mean():.2f}")


Graph Structure:
First 10 edges (src -> dst):
  0 -> 0
  0 -> 1
  0 -> 2
  0 -> 3
  0 -> 4
  0 -> 5
  0 -> 6
  0 -> 7
  0 -> 8
  0 -> 9

Number of self-loops: 10

Degree statistics:
  In-degree  - Min: 10, Max: 10, Mean: 10.00
  Out-degree - Min: 10, Max: 10, Mean: 10.00


In [24]:
# Statistics across all graphs
print("\nStatistics Across All Graphs:")
print("="*50)

num_nodes_list = [g.num_nodes() for g in graphs.values()]
num_edges_list = [g.num_edges() for g in graphs.values()]

print(f"Number of nodes:")
print(f"  Min: {min(num_nodes_list)}, Max: {max(num_nodes_list)}, Mean: {np.mean(num_nodes_list):.2f}")
print(f"\nNumber of edges:")
print(f"  Min: {min(num_edges_list)}, Max: {max(num_edges_list)}, Mean: {np.mean(num_edges_list):.2f}")

# Check if all graphs have the same structure
all_same_nodes = len(set(num_nodes_list)) == 1
all_same_edges = len(set(num_edges_list)) == 1
print(f"\nAll graphs have same number of nodes: {all_same_nodes}")
print(f"All graphs have same number of edges: {all_same_edges}")


Statistics Across All Graphs:
Number of nodes:
  Min: 10, Max: 10, Mean: 10.00

Number of edges:
  Min: 100, Max: 100, Mean: 100.00

All graphs have same number of nodes: True
All graphs have same number of edges: True


In [25]:
# How to access specific graphs
print("\nAccessing Specific Graphs:")
print("="*50)

# Access by graph ID
specific_ids = ['0_0', '21_5', '49_9']
for graph_id in specific_ids:
    if graph_id in graphs:
        g = graphs[graph_id]
        print(f"\nGraph {graph_id}:")
        print(f"  Nodes: {g.num_nodes()}, Edges: {g.num_edges()}")
        print(f"  R0 values: {g.ndata['R0'].numpy()}")
    else:
        print(f"\nGraph {graph_id}: Not found")


Accessing Specific Graphs:

Graph 0_0:
  Nodes: 10, Edges: 100
  R0 values: [3.2351675 4.0134044 3.9334474 3.213428  3.3135781 2.9970555 2.9382489
 3.478308  3.3988512 3.3105624]

Graph 21_5:
  Nodes: 10, Edges: 100
  R0 values: [4.671173  3.8996673 5.2221217 3.7213082 3.6686385 5.1829934 4.830035
 4.874202  4.32116   4.530192 ]

Graph 49_9:
  Nodes: 10, Edges: 100
  R0 values: [3.1825864 2.4432507 3.3714275 3.0560105 2.2110052 2.7159388 4.0805044
 2.3213465 3.2321942 2.6729424]
